## Creae Fact Table

** Reading silver Data**

In [0]:

df_silver= spark.sql('''
select * from parquet.`abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata`''')

df_silver.display()

# Reading all Dim table

In [0]:
df_branch =  spark.sql("select * from car_project.gold.dim_branch")
df_model = spark.sql("select * from car_project.gold.dim_model")
df_date = spark.sql("select * from car_project.gold.dim_date")
df_dealer = spark.sql("select * from car_project.gold.dim_dealer")

# Joining All dim table with fact table

In [0]:
df_fact = (
    df_silver
    .join(df_branch, df_branch["Branch_ID"] == df_silver["Branch_id"], "left")
    .join(df_model, df_model["Model_Id"] == df_silver["Model_ID"], "left")
    .join(df_date, df_date["Date_ID"] == df_silver["Date_ID"], "left")
    .join(df_dealer, df_dealer["Dealer_ID"] == df_silver["Dealer_id"], "left")
    .select(
        df_silver["Revenue"],
        df_silver["Units_Sold"],
        df_silver["Single_unit_cost"],
        df_branch["dim_branch_key"],
        df_date["dim_date_key"],
        df_dealer["dim_dealer_key"],
        df_model["dim_model_key"]
    )
)

In [0]:
df_fact.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
# Incremental RUN
if spark.catalog.tableExists("car_project.gold.factsales"):
    delta_table = DeltaTable.forPath(spark,"car_project.gold.factsales")
    delta_table.alias("target").merge(df_fact.alias("source"),"target.dim_branch_key = source.dim_branch_key and target.dim_dealer_key and  source.dim_dealer_key and target.dim_model_key = source.dim_model_key and target.dim_date_key and source.dim_date_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    
else:
    df_fact.write.format("delta") \
        .mode("append") \
        .option("path", "abfss://gold-layer@adlcarproject.dfs.core.windows.net/factsales") \
        .saveAsTable("car_project.gold.factsales")

In [0]:
%sql
select * from car_project.gold.factsales